> **Note**: This notebook performs a **complete COCO → YOLO conversion** of the TACO dataset, including **ALL categories** present in the COCO annotation file. No categories are filtered, merged, or discarded. The output is a general Waste Detection YOLO dataset suitable for training YOLOv11.

# 🔄 TACO COCO → YOLO Complete Dataset Conversion

## Overview
This notebook converts the **TACO (Trash Annotations in Context)** dataset from **COCO annotation format** to **YOLO format**, preserving **every category** present in the original COCO JSON.

### Key Design Principles
- **ALL categories** are read dynamically from the COCO JSON — no hardcoded class lists
- **ALL valid annotations** are converted — no filtering by category name
- COCO category IDs are mapped to **contiguous YOLO class IDs** (0..N-1)
- Bounding boxes are **validated and clipped** to image boundaries
- Invalid annotations are **reported**, not silently discarded
- The original TACO dataset is **never modified**

### Pipeline
```
TACO COCO JSON → Read ALL Categories → Map to YOLO IDs → Convert BBoxes → Split 80/10/10 → YOLO Dataset
```

## 1. Environment Setup & Library Imports

In [ ]:
!pip install -q pycocotools rich tqdm scikit-learn pandas opencv-python matplotlib pillow pyyaml

In [ ]:
import os
import json
import csv
import shutil
import random
import math
import warnings
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import yaml
from tqdm import tqdm
from pycocotools.coco import COCO
from sklearn.model_selection import train_test_split
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.tree import Tree

console = Console()
warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

## 2. Load and Validate Dataset
Load both the pycocotools COCO object and the raw JSON to ensure we have direct access to all categories, images, and annotations.

In [ ]:
# ==========================================
# Configuration — Adjust paths as needed
# ==========================================
PROJECT_ROOT = Path('/content/drive/MyDrive/PlasticSense_AI')
ANN_FILE = PROJECT_ROOT / 'datasets/taco/raw/TACO/data/annotations.json'
ACTUAL_IMAGES_BASE = PROJECT_ROOT / 'datasets/taco/raw/TACO/data'

# Output directory — new dataset, does NOT overwrite original
OUTPUT_DIR = PROJECT_ROOT / 'datasets/taco_yolo_all_categories'
REPORTS_DIR = OUTPUT_DIR / 'reports'

# ==========================================
# Validate source files exist
# ==========================================
if not ANN_FILE.exists():
    raise FileNotFoundError(f"Annotations file not found at: {ANN_FILE}")
console.print("[green]✔ TACO Annotations file located successfully.[/green]")

# ==========================================
# Load raw JSON for direct access
# ==========================================
with open(ANN_FILE, 'r') as f:
    coco_data = json.load(f)

# ==========================================
# Also load via pycocotools for convenience
# ==========================================
console.print("[cyan]Loading COCO Annotations via pycocotools...[/cyan]")
coco = COCO(str(ANN_FILE))

# ==========================================
# Print dataset statistics from raw JSON
# ==========================================
num_images = len(coco_data.get('images', []))
num_annotations = len(coco_data.get('annotations', []))
num_categories = len(coco_data.get('categories', []))

console.print(Panel.fit(
    f"[bold cyan]TACO Dataset Summary[/bold cyan]\n\n"
    f"Images: [bold]{num_images}[/bold]\n"
    f"Annotations: [bold]{num_annotations}[/bold]\n"
    f"Categories: [bold]{num_categories}[/bold]"
))

## 3. Dynamic Category Mapping — ALL Categories

**CRITICAL**: Every category in `coco_data["categories"]` is included. No filtering, no merging, no discarding.

COCO category IDs may not be contiguous, so we create a deterministic mapping:
- Sort categories by their original COCO `id`
- Assign contiguous YOLO class IDs: 0, 1, 2, ..., N-1

In [ ]:
# ==========================================
# Read ALL categories from the COCO JSON
# ==========================================
categories = coco_data["categories"]
console.print(f"[cyan]Found {len(categories)} categories in COCO JSON.[/cyan]")

# Sort by COCO category ID for deterministic ordering
categories_sorted = sorted(categories, key=lambda c: c['id'])

# ==========================================
# Build COCO ID → YOLO ID mapping
# ==========================================
coco_to_yolo_map = {}   # COCO category_id → YOLO class_id
yolo_class_names = {}    # YOLO class_id → category name
yolo_to_coco_map = {}    # YOLO class_id → COCO category_id

for yolo_id, cat in enumerate(categories_sorted):
    coco_id = cat['id']
    cat_name = cat['name']
    coco_to_yolo_map[coco_id] = yolo_id
    yolo_class_names[yolo_id] = cat_name
    yolo_to_coco_map[yolo_id] = coco_id

total_categories = len(yolo_class_names)
console.print(f"[green]✔ Mapped {total_categories} categories to contiguous YOLO IDs (0..{total_categories - 1})[/green]")

# ==========================================
# Count annotations and images per category
# ==========================================
ann_count_per_coco_id = defaultdict(int)
img_set_per_coco_id = defaultdict(set)

for ann in coco_data['annotations']:
    cat_id = ann.get('category_id')
    img_id = ann.get('image_id')
    if cat_id is not None:
        ann_count_per_coco_id[cat_id] += 1
    if cat_id is not None and img_id is not None:
        img_set_per_coco_id[cat_id].add(img_id)

# ==========================================
# Display complete category table
# ==========================================
table = Table(title="Complete Category Mapping (ALL Categories)", show_header=True, show_lines=True)
table.add_column("COCO ID", style="cyan", justify="right")
table.add_column("YOLO ID", style="green", justify="right")
table.add_column("Category", style="bold white")
table.add_column("Annotations", justify="right")
table.add_column("Images", justify="right")

for yolo_id in range(total_categories):
    coco_id = yolo_to_coco_map[yolo_id]
    cat_name = yolo_class_names[yolo_id]
    ann_count = ann_count_per_coco_id.get(coco_id, 0)
    img_count = len(img_set_per_coco_id.get(coco_id, set()))
    table.add_row(str(coco_id), str(yolo_id), cat_name, str(ann_count), str(img_count))

console.print(table)

# ==========================================
# Save category_mapping.csv
# ==========================================
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

mapping_rows = []
for yolo_id in range(total_categories):
    coco_id = yolo_to_coco_map[yolo_id]
    mapping_rows.append({
        'coco_category_id': coco_id,
        'yolo_class_id': yolo_id,
        'category_name': yolo_class_names[yolo_id]
    })

df_mapping = pd.DataFrame(mapping_rows)
mapping_csv_path = OUTPUT_DIR / 'category_mapping.csv'
df_mapping.to_csv(mapping_csv_path, index=False)
console.print(f"[green]✔ Saved category_mapping.csv ({len(mapping_rows)} categories)[/green]")

### Central Class Configuration — `configs/classes.yaml`
Generate a single source of truth for class mapping that ALL downstream notebooks will use. This prevents class-ID mismatch between notebooks.

In [ ]:
# ==========================================
# Generate configs/classes.yaml — Single Source of Truth
# ==========================================
CONFIGS_DIR = PROJECT_ROOT / 'configs'
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

classes_yaml_data = {
    'total_categories': total_categories,
    'source_dataset': 'TACO',
    'source_annotation': str(ANN_FILE),
    'categories': []
}

# High-level waste group mapping (for analysis/grouping only)
def _classify_waste_group(cat_name: str) -> str:
    name = cat_name.lower()
    if any(kw in name for kw in ['plastic', 'bottle', 'straw', 'cup', 'lid', 'wrapper',
                                   'blister', 'film', 'polystyrene', 'styrofoam',
                                   'tupperware', 'squeezable', 'six pack', 'spread tub']):
        if 'glass' in name: return 'glass'
        if 'metal' in name or 'aluminium' in name or 'aluminum' in name: return 'metal'
        return 'plastic'
    if 'glass' in name: return 'glass'
    if any(kw in name for kw in ['metal', 'aluminium', 'aluminum', 'can', 'tin', 'foil', 'aerosol']): return 'metal'
    if any(kw in name for kw in ['paper', 'magazine', 'newspaper', 'tissues', 'napkins']): return 'paper'
    if any(kw in name for kw in ['cardboard', 'carton', 'egg carton', 'pizza box']): return 'cardboard'
    if any(kw in name for kw in ['foam', 'styrofoam', 'polystyrene']): return 'foam'
    if any(kw in name for kw in ['clothing', 'textile', 'shoe', 'fabric']): return 'textile'
    if any(kw in name for kw in ['rubber', 'tire']): return 'rubber'
    if any(kw in name for kw in ['wood', 'cork']): return 'wood'
    if any(kw in name for kw in ['food', 'organic']): return 'organic'
    if 'cigarette' in name: return 'other'
    if any(kw in name for kw in ['battery', 'electronic', 'rope', 'net']): return 'other'
    return 'unknown'

for yolo_id in range(total_categories):
    coco_id = yolo_to_coco_map[yolo_id]
    cat_name = yolo_class_names[yolo_id]
    waste_group = _classify_waste_group(cat_name)
    classes_yaml_data['categories'].append({
        'coco_id': int(coco_id),
        'yolo_id': int(yolo_id),
        'name': cat_name,
        'waste_group': waste_group
    })

classes_yaml_path = CONFIGS_DIR / 'classes.yaml'
with open(classes_yaml_path, 'w') as f:
    yaml.dump(classes_yaml_data, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

console.print(f"[green]✔ Saved configs/classes.yaml ({total_categories} categories)[/green]")
console.print(f"[cyan]  Path: {classes_yaml_path}[/cyan]")
console.print("[cyan]  This file is the SINGLE SOURCE OF TRUTH for all downstream notebooks.[/cyan]")

## 4. Category Distribution Analysis
Compute annotation counts, image counts, and percentages for every category. Visualize with a bar chart.

In [ ]:
# ==========================================
# Build category distribution DataFrame
# ==========================================
total_ann_count = len(coco_data['annotations'])

dist_rows = []
for yolo_id in range(total_categories):
    coco_id = yolo_to_coco_map[yolo_id]
    cat_name = yolo_class_names[yolo_id]
    ann_count = ann_count_per_coco_id.get(coco_id, 0)
    img_count = len(img_set_per_coco_id.get(coco_id, set()))
    pct = (ann_count / total_ann_count * 100) if total_ann_count > 0 else 0.0
    dist_rows.append({
        'class_id': yolo_id,
        'coco_category_id': coco_id,
        'category_name': cat_name,
        'annotation_count': ann_count,
        'image_count': img_count,
        'percentage_of_annotations': round(pct, 2)
    })

df_dist = pd.DataFrame(dist_rows)
dist_csv_path = OUTPUT_DIR / 'category_distribution.csv'
df_dist.to_csv(dist_csv_path, index=False)
console.print(f"[green]✔ Saved category_distribution.csv[/green]")

# ==========================================
# Bar chart of annotation counts
# ==========================================
fig, ax = plt.subplots(figsize=(max(14, total_categories * 0.4), 8))
bars = ax.barh(
    [f"{r['category_name']} (ID:{r['class_id']})" for r in dist_rows],
    [r['annotation_count'] for r in dist_rows],
    color=plt.cm.viridis(np.linspace(0.2, 0.9, total_categories))
)
ax.set_xlabel('Annotation Count', fontsize=12)
ax.set_title('Annotation Distribution Across ALL Categories', fontsize=14, fontweight='bold')
ax.invert_yaxis()

# Add count labels
for bar, row in zip(bars, dist_rows):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(row['annotation_count']), va='center', fontsize=8)

plt.tight_layout()
plt.show()

console.print(f"[cyan]Total annotations: {total_ann_count}[/cyan]")

## 5. COCO to YOLO Bounding Box Conversion

Converts COCO format `[x, y, width, height]` (top-left corner) to YOLO format `[x_center, y_center, width, height]` (normalized center).

Includes proper **clipping** — bounding boxes that extend outside the image are clipped to image boundaries before normalization.

In [ ]:
def coco_to_yolo_bbox(bbox: List[float], img_w: int, img_h: int) -> Optional[Tuple[float, float, float, float]]:
    """
    Convert a COCO bounding box to YOLO format with clipping.

    COCO: [x_top_left, y_top_left, box_width, box_height] (pixels)
    YOLO: [x_center, y_center, width, height] (normalized 0-1)

    Returns None if the bbox is invalid after clipping.
    """
    x, y, w, h = bbox

    # Reject zero/negative dimensions
    if w <= 0 or h <= 0:
        return None

    # Reject invalid image dimensions
    if img_w <= 0 or img_h <= 0:
        return None

    # Clip bounding box to image boundaries
    x1 = max(0.0, x)
    y1 = max(0.0, y)
    x2 = min(float(img_w), x + w)
    y2 = min(float(img_h), y + h)

    # Recalculate width/height after clipping
    clipped_w = x2 - x1
    clipped_h = y2 - y1

    # Reject if clipping eliminated the box
    if clipped_w <= 0 or clipped_h <= 0:
        return None

    # Calculate center coordinates
    x_center = x1 + clipped_w / 2.0
    y_center = y1 + clipped_h / 2.0

    # Normalize to [0, 1]
    x_center /= img_w
    y_center /= img_h
    norm_w = clipped_w / img_w
    norm_h = clipped_h / img_h

    # Final validation
    if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < norm_w <= 1 and 0 < norm_h <= 1):
        return None

    return x_center, y_center, norm_w, norm_h

console.print("[green]✔ COCO → YOLO bbox conversion function defined (with clipping & validation)[/green]")

## 6. Process ALL Annotations & Build Dataset

Process **every** annotation in the COCO JSON:
1. Map COCO category ID → YOLO class ID
2. Convert bounding box to YOLO format (with clipping)
3. Track invalid annotations with reasons
4. Build per-image label data for ALL images

**No categories are filtered or skipped.**

In [ ]:
# ==========================================
# Build image lookup: image_id → image info
# ==========================================
image_lookup = {}
for img in coco_data['images']:
    image_lookup[img['id']] = img

# ==========================================
# Build annotation lookup: image_id → [annotations]
# ==========================================
annotations_by_image = defaultdict(list)
for ann in coco_data['annotations']:
    annotations_by_image[ann['image_id']].append(ann)

# ==========================================
# Process ALL annotations
# ==========================================
invalid_annotations = []   # For invalid_annotations.csv
image_labels = {}          # image_id → {'img_info': ..., 'img_path': ..., 'yolo_lines': [...]}
images_not_found = set()
valid_ann_count = 0
total_ann_processed = 0

console.print(f"[cyan]Processing {len(coco_data['annotations'])} annotations across {len(coco_data['images'])} images...[/cyan]")
console.print(f"[cyan]Searching for images in: {ACTUAL_IMAGES_BASE}[/cyan]")

for img_info in tqdm(coco_data['images'], desc="Processing images"):
    img_id = img_info['id']
    img_w = img_info.get('width', 0)
    img_h = img_info.get('height', 0)
    file_name = img_info.get('file_name', '')

    # Resolve image path
    img_path = ACTUAL_IMAGES_BASE / file_name
    if not img_path.exists():
        # Fallback: try just the filename
        img_path = ACTUAL_IMAGES_BASE / Path(file_name).name
        if not img_path.exists():
            images_not_found.add(img_id)
            # Record all annotations for this image as invalid
            for ann in annotations_by_image.get(img_id, []):
                total_ann_processed += 1
                cat_id = ann.get('category_id', -1)
                cat_name = yolo_class_names.get(coco_to_yolo_map.get(cat_id, -1), 'unknown')
                invalid_annotations.append({
                    'annotation_id': ann.get('id', ''),
                    'image_id': img_id,
                    'category_id': cat_id,
                    'category_name': cat_name,
                    'bbox': str(ann.get('bbox', [])),
                    'reason': 'missing image'
                })
            continue

    # Validate image dimensions
    if img_w <= 0 or img_h <= 0:
        for ann in annotations_by_image.get(img_id, []):
            total_ann_processed += 1
            cat_id = ann.get('category_id', -1)
            cat_name = yolo_class_names.get(coco_to_yolo_map.get(cat_id, -1), 'unknown')
            invalid_annotations.append({
                'annotation_id': ann.get('id', ''),
                'image_id': img_id,
                'category_id': cat_id,
                'category_name': cat_name,
                'bbox': str(ann.get('bbox', [])),
                'reason': 'invalid image dimensions'
            })
        continue

    # Initialize image entry
    if img_id not in image_labels:
        image_labels[img_id] = {
            'img_info': img_info,
            'img_path': img_path,
            'yolo_lines': []
        }

    # Process each annotation for this image
    for ann in annotations_by_image.get(img_id, []):
        total_ann_processed += 1
        cat_id = ann.get('category_id')
        ann_id = ann.get('id', '')
        bbox = ann.get('bbox')

        # Check for unknown category
        if cat_id not in coco_to_yolo_map:
            cat_name = f'unknown_cat_{cat_id}'
            invalid_annotations.append({
                'annotation_id': ann_id,
                'image_id': img_id,
                'category_id': cat_id,
                'category_name': cat_name,
                'bbox': str(bbox),
                'reason': 'unknown category'
            })
            continue

        yolo_class_id = coco_to_yolo_map[cat_id]
        cat_name = yolo_class_names[yolo_class_id]

        # Check for missing/malformed bbox
        if bbox is None or not isinstance(bbox, list) or len(bbox) != 4:
            invalid_annotations.append({
                'annotation_id': ann_id,
                'image_id': img_id,
                'category_id': cat_id,
                'category_name': cat_name,
                'bbox': str(bbox),
                'reason': 'malformed annotation'
            })
            continue

        # Check for zero dimensions before conversion
        if bbox[2] <= 0:
            invalid_annotations.append({
                'annotation_id': ann_id,
                'image_id': img_id,
                'category_id': cat_id,
                'category_name': cat_name,
                'bbox': str(bbox),
                'reason': 'zero width'
            })
            continue

        if bbox[3] <= 0:
            invalid_annotations.append({
                'annotation_id': ann_id,
                'image_id': img_id,
                'category_id': cat_id,
                'category_name': cat_name,
                'bbox': str(bbox),
                'reason': 'zero height'
            })
            continue

        # Convert to YOLO format (with clipping)
        result = coco_to_yolo_bbox(bbox, img_w, img_h)

        if result is None:
            invalid_annotations.append({
                'annotation_id': ann_id,
                'image_id': img_id,
                'category_id': cat_id,
                'category_name': cat_name,
                'bbox': str(bbox),
                'reason': 'invalid coordinates'
            })
            continue

        x_c, y_c, w_n, h_n = result
        yolo_line = f"{yolo_class_id} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}"
        image_labels[img_id]['yolo_lines'].append(yolo_line)
        valid_ann_count += 1

# Also register images that have NO annotations at all
for img_info in coco_data['images']:
    img_id = img_info['id']
    if img_id in image_labels or img_id in images_not_found:
        continue
    file_name = img_info.get('file_name', '')
    img_path = ACTUAL_IMAGES_BASE / file_name
    if not img_path.exists():
        img_path = ACTUAL_IMAGES_BASE / Path(file_name).name
    if img_path.exists():
        image_labels[img_id] = {
            'img_info': img_info,
            'img_path': img_path,
            'yolo_lines': []
        }

console.print(Panel.fit(
    f"[bold cyan]Annotation Processing Summary[/bold cyan]\n\n"
    f"Total annotations processed: [bold]{total_ann_processed}[/bold]\n"
    f"Valid annotations: [bold green]{valid_ann_count}[/bold green]\n"
    f"Invalid annotations: [bold red]{len(invalid_annotations)}[/bold red]\n"
    f"Images with resolved paths: [bold]{len(image_labels)}[/bold]\n"
    f"Images not found: [bold yellow]{len(images_not_found)}[/bold yellow]"
))

## 7. Invalid Annotation Report
Document all annotations that could not be converted, with reasons.

In [ ]:
# ==========================================
# Save invalid_annotations.csv
# ==========================================
invalid_csv_path = OUTPUT_DIR / 'invalid_annotations.csv'

if invalid_annotations:
    df_invalid = pd.DataFrame(invalid_annotations)
    df_invalid.to_csv(invalid_csv_path, index=False)
    console.print(f"[yellow]⚠ Saved {len(invalid_annotations)} invalid annotations to invalid_annotations.csv[/yellow]")

    # Show breakdown by reason
    reason_counts = df_invalid['reason'].value_counts()
    table = Table(title="Invalid Annotation Reasons", show_header=True)
    table.add_column("Reason", style="red")
    table.add_column("Count", justify="right")
    for reason, count in reason_counts.items():
        table.add_row(reason, str(count))
    console.print(table)
else:
    # Write empty CSV with headers
    pd.DataFrame(columns=['annotation_id', 'image_id', 'category_id', 'category_name', 'bbox', 'reason']).to_csv(invalid_csv_path, index=False)
    console.print("[green]✔ No invalid annotations found![/green]")

console.print(f"\n[bold]Total annotations: {total_ann_processed}[/bold]")
console.print(f"[bold green]Valid annotations: {valid_ann_count}[/bold green]")
console.print(f"[bold red]Invalid annotations: {len(invalid_annotations)}[/bold red]")

## 8. Images Without Valid Annotations
Identify images that have no annotations or only invalid annotations. These are retained as **negative examples** (empty label files).

In [ ]:
# ==========================================
# Find images with no valid YOLO labels
# ==========================================
images_without_valid_annotations = []

for img_id, data in image_labels.items():
    if len(data['yolo_lines']) == 0:
        img_info = data['img_info']
        images_without_valid_annotations.append({
            'image_id': img_id,
            'file_name': img_info.get('file_name', ''),
            'width': img_info.get('width', 0),
            'height': img_info.get('height', 0),
            'total_annotations_in_coco': len(annotations_by_image.get(img_id, [])),
            'reason': 'no valid annotations' if img_id in annotations_by_image else 'no annotations'
        })

no_ann_csv_path = OUTPUT_DIR / 'images_without_annotations.csv'
df_no_ann = pd.DataFrame(images_without_valid_annotations)
df_no_ann.to_csv(no_ann_csv_path, index=False)

console.print(f"[cyan]Images without valid annotations: {len(images_without_valid_annotations)}[/cyan]")
console.print("[cyan]These images are retained as negative examples (empty .txt label files).[/cyan]")
console.print(f"[green]✔ Saved images_without_annotations.csv[/green]")

## 9. Train / Validation / Test Split (80/10/10)
Split is performed at the **image level**. Each image appears in exactly one split. Deterministic with `random_state=42`.

In [ ]:
# ==========================================
# Prepare image list for splitting
# ==========================================
all_image_ids = list(image_labels.keys())
console.print(f"[cyan]Total images for splitting: {len(all_image_ids)}[/cyan]")

# ==========================================
# 80/10/10 split
# ==========================================
train_ids, temp_ids = train_test_split(all_image_ids, test_size=0.20, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=42)

split_map = {}
for img_id in train_ids:
    split_map[img_id] = 'train'
for img_id in val_ids:
    split_map[img_id] = 'val'
for img_id in test_ids:
    split_map[img_id] = 'test'

console.print(Panel.fit(
    f"[bold cyan]Dataset Split[/bold cyan]\n\n"
    f"Train: [bold]{len(train_ids)}[/bold] images ({len(train_ids)/len(all_image_ids)*100:.1f}%)\n"
    f"Val:   [bold]{len(val_ids)}[/bold] images ({len(val_ids)/len(all_image_ids)*100:.1f}%)\n"
    f"Test:  [bold]{len(test_ids)}[/bold] images ({len(test_ids)/len(all_image_ids)*100:.1f}%)\n"
    f"Total: [bold]{len(all_image_ids)}[/bold] images"
))

## 10. Create YOLO Dataset Directory & Write Files
Create the directory structure, copy images, and write label files.

In [ ]:
# ==========================================
# Create directory structure
# ==========================================
for split_name in ['train', 'val', 'test']:
    img_dir = OUTPUT_DIR / 'images' / split_name
    lbl_dir = OUTPUT_DIR / 'labels' / split_name

    # Clean existing to prevent stale files
    if img_dir.exists():
        shutil.rmtree(img_dir)
    if lbl_dir.exists():
        shutil.rmtree(lbl_dir)

    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

console.print("[green]✔ Directory structure created[/green]")

# ==========================================
# Copy images and write label files
# ==========================================
total_images_copied = 0
total_labels_written = 0
total_annotations_written = 0

for img_id, data in tqdm(image_labels.items(), desc="Writing YOLO dataset"):
    split_name = split_map.get(img_id)
    if split_name is None:
        continue

    img_path = data['img_path']
    yolo_lines = data['yolo_lines']
    img_info = data['img_info']

    # Create a unique filename to avoid collisions
    # Use parent directory name + original stem
    parent_name = img_path.parent.name
    ext = img_path.suffix.lower()
    if ext not in ['.jpg', '.jpeg', '.png']:
        ext = '.jpg'
    unique_name = f"{parent_name}_{img_path.stem}{ext}"

    img_dest = OUTPUT_DIR / 'images' / split_name / unique_name
    lbl_dest = OUTPUT_DIR / 'labels' / split_name / f"{Path(unique_name).stem}.txt"

    # Copy image (do not modify)
    shutil.copy2(str(img_path), str(img_dest))
    total_images_copied += 1

    # Write label file (may be empty for negative examples)
    with open(lbl_dest, 'w') as f:
        if yolo_lines:
            f.write("\n".join(yolo_lines))
            total_annotations_written += len(yolo_lines)
    total_labels_written += 1

console.print(Panel.fit(
    f"[bold green]✔ YOLO Dataset Written[/bold green]\n\n"
    f"Images copied: [bold]{total_images_copied}[/bold]\n"
    f"Label files created: [bold]{total_labels_written}[/bold]\n"
    f"Annotations written: [bold]{total_annotations_written}[/bold]"
))

## 11. Generate `data.yaml`
The YOLO configuration file. `nc` and `names` are generated **dynamically** from the category mapping — never hardcoded.

In [ ]:
# ==========================================
# Build data.yaml content
# ==========================================
data_yaml = {
    'path': str(OUTPUT_DIR.absolute()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': total_categories,
    'names': {yolo_id: yolo_class_names[yolo_id] for yolo_id in range(total_categories)}
}

yaml_path = OUTPUT_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

# Display the generated YAML
with open(yaml_path, 'r') as f:
    yaml_content = f.read()

console.print(Panel.fit(
    f"[bold green]✔ data.yaml Generated[/bold green]\n\n"
    f"[cyan]{yaml_content}[/cyan]"
))

# Verify nc matches
assert data_yaml['nc'] == total_categories, f"nc mismatch: {data_yaml['nc']} != {total_categories}"
assert len(data_yaml['names']) == total_categories, f"names count mismatch: {len(data_yaml['names'])} != {total_categories}"
console.print(f"[green]✔ Verified: nc={total_categories}, names count={len(data_yaml['names'])}[/green]")

## 12. Comprehensive Dataset Validation
Perform all 13 validation checks to ensure dataset integrity.

In [ ]:
def validate_yolo_dataset(yolo_dir: Path, class_names: dict, num_classes: int):
    """Comprehensive 13-point YOLO dataset validation."""
    results = {}
    issues = []
    total_annotations = 0
    class_counts = defaultdict(int)

    # ---- Check 1: nc matches category count ----
    results['nc_matches'] = (num_classes == len(class_names))
    if not results['nc_matches']:
        issues.append(f"nc ({num_classes}) != len(names) ({len(class_names)})")

    # ---- Check 2: names count matches nc ----
    results['names_count_matches'] = (len(class_names) == num_classes)

    # ---- Check 3: Class IDs are contiguous 0..N-1 ----
    expected_ids = set(range(num_classes))
    actual_ids = set(class_names.keys())
    results['class_ids_contiguous'] = (expected_ids == actual_ids)
    if not results['class_ids_contiguous']:
        missing = expected_ids - actual_ids
        extra = actual_ids - expected_ids
        if missing:
            issues.append(f"Missing class IDs: {missing}")
        if extra:
            issues.append(f"Extra class IDs: {extra}")

    # ---- Check 4-12: Per-split validation ----
    results['all_labels_valid'] = True
    results['all_bboxes_valid'] = True
    results['all_bboxes_normalized'] = True
    results['all_images_exist'] = True
    results['no_missing_labels'] = True
    results['no_unknown_class_ids'] = True

    split_stats = {}
    for split in ['train', 'val', 'test']:
        img_dir = yolo_dir / 'images' / split
        lbl_dir = yolo_dir / 'labels' / split

        # Check 13: folders exist
        if not img_dir.exists() or not lbl_dir.exists():
            issues.append(f"Missing directory for {split} split")
            continue

        imgs = sorted(list(img_dir.glob('*.[jJ][pP][gG]')) +
                       list(img_dir.glob('*.[jJ][pP][eE][gG]')) +
                       list(img_dir.glob('*.[pP][nN][gG]')))
        lbls = sorted(list(lbl_dir.glob('*.txt')))

        split_stats[split] = {'images': len(imgs), 'labels': len(lbls)}

        # Check image-label correspondence
        img_stems = {p.stem for p in imgs}
        lbl_stems = {p.stem for p in lbls}

        missing_labels = img_stems - lbl_stems
        if missing_labels:
            results['no_missing_labels'] = False
            issues.append(f"{split}: {len(missing_labels)} images without labels")

        # Validate each label file
        for lbl in lbls:
            with open(lbl, 'r') as f:
                lines = f.readlines()

            for line_num, line in enumerate(lines, 1):
                line = line.strip()
                if not line:
                    continue

                parts = line.split()

                # Check 5: exactly 5 values
                if len(parts) != 5:
                    results['all_labels_valid'] = False
                    issues.append(f"{split}/{lbl.name}:{line_num} has {len(parts)} values (expected 5)")
                    continue

                try:
                    cls_id = int(parts[0])
                    coords = [float(p) for p in parts[1:]]
                except ValueError:
                    results['all_labels_valid'] = False
                    issues.append(f"{split}/{lbl.name}:{line_num} has non-numeric values")
                    continue

                # Check 4: valid class ID
                if cls_id < 0 or cls_id >= num_classes:
                    results['no_unknown_class_ids'] = False
                    issues.append(f"{split}/{lbl.name}:{line_num} unknown class ID {cls_id}")

                # Check 6: normalized [0,1]
                x_c, y_c, w, h = coords
                if not (0 <= x_c <= 1 and 0 <= y_c <= 1):
                    results['all_bboxes_normalized'] = False
                    issues.append(f"{split}/{lbl.name}:{line_num} center out of [0,1]")

                # Check 7: valid bbox
                if not (0 < w <= 1 and 0 < h <= 1):
                    results['all_bboxes_valid'] = False
                    issues.append(f"{split}/{lbl.name}:{line_num} invalid bbox dimensions")

                class_counts[cls_id] += 1
                total_annotations += 1

    # ---- Check 12: data.yaml is valid ----
    yaml_path = yolo_dir / 'data.yaml'
    results['data_yaml_valid'] = yaml_path.exists()
    if yaml_path.exists():
        with open(yaml_path, 'r') as f:
            try:
                loaded_yaml = yaml.safe_load(f)
                if loaded_yaml.get('nc') != num_classes:
                    results['data_yaml_valid'] = False
                    issues.append(f"data.yaml nc mismatch")
            except Exception as e:
                results['data_yaml_valid'] = False
                issues.append(f"data.yaml parse error: {e}")

    # ---- Check 13: train/val/test folders exist ----
    results['folders_exist'] = all(
        (yolo_dir / 'images' / s).exists() and (yolo_dir / 'labels' / s).exists()
        for s in ['train', 'val', 'test']
    )

    # ---- Print validation report ----
    console.print("\n[bold]" + "=" * 50 + "[/bold]")
    console.print("[bold cyan]          DATASET VALIDATION[/bold cyan]")
    console.print("[bold]" + "=" * 50 + "[/bold]")

    for split, stats in split_stats.items():
        console.print(f"  {split.capitalize():6s} — Images: {stats['images']}, Labels: {stats['labels']}")

    console.print(f"\n  Categories:        {num_classes}")
    console.print(f"  Valid annotations: {total_annotations}")
    console.print(f"  Invalid (earlier): {len(invalid_annotations)}")

    checks = [
        ('Class IDs valid',      results.get('class_ids_contiguous', False)),
        ('Labels valid',         results.get('all_labels_valid', False)),
        ('Bounding boxes valid', results.get('all_bboxes_valid', False) and results.get('all_bboxes_normalized', False)),
        ('Images valid',         results.get('all_images_exist', False)),
        ('No missing labels',    results.get('no_missing_labels', False)),
        ('No unknown class IDs', results.get('no_unknown_class_ids', False)),
        ('data.yaml valid',      results.get('data_yaml_valid', False)),
        ('Folders exist',        results.get('folders_exist', False)),
    ]

    console.print("")
    all_pass = True
    for name, passed in checks:
        status = "[bold green]PASS[/bold green]" if passed else "[bold red]FAIL[/bold red]"
        if not passed:
            all_pass = False
        console.print(f"  {name:25s} {status}")

    final = "[bold green]PASS[/bold green]" if all_pass else "[bold red]FAIL[/bold red]"
    console.print(f"\n  [bold]FINAL STATUS: {final}[/bold]")
    console.print("[bold]" + "=" * 50 + "[/bold]")

    if issues:
        console.print(f"\n[yellow]Issues found ({len(issues)}):[/yellow]")
        for iss in issues[:20]:
            console.print(f"  [yellow]• {iss}[/yellow]")
        if len(issues) > 20:
            console.print(f"  [yellow]... and {len(issues) - 20} more[/yellow]")

    return class_counts, total_annotations, issues, all_pass

# Run validation
val_class_counts, val_total_anns, val_issues, val_passed = validate_yolo_dataset(
    OUTPUT_DIR, yolo_class_names, total_categories
)

## 13. Visual Annotation Check
Visualize YOLO annotations on images to verify conversion correctness. Intentionally includes images with multiple objects and categories.

In [ ]:
def visualize_yolo_annotations(yolo_dir: Path, class_names: dict, num_images: int = 12, split: str = 'train'):
    """
    Visualize YOLO annotations on images.

    Selects images that have the most annotations first (to show dense/multi-category examples),
    then fills remaining slots randomly.
    """
    img_dir = yolo_dir / 'images' / split
    lbl_dir = yolo_dir / 'labels' / split

    if not img_dir.exists():
        print(f"Error: Directory {img_dir} does not exist.")
        return

    # Find all images
    all_imgs = sorted(
        list(img_dir.glob('*.[jJ][pP][gG]')) +
        list(img_dir.glob('*.[jJ][pP][eE][gG]')) +
        list(img_dir.glob('*.[pP][nN][gG]'))
    )

    if not all_imgs:
        print("No images found for visualization.")
        return

    # Score images by annotation count & category diversity
    scored = []
    for img_path in all_imgs:
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        ann_count = 0
        cat_set = set()
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        ann_count += 1
                        cat_set.add(int(parts[0]))
        scored.append((img_path, ann_count, len(cat_set)))

    # Sort by annotation count descending, then by category diversity
    scored.sort(key=lambda x: (x[1], x[2]), reverse=True)

    # Take top examples + some random ones
    n_dense = min(num_images // 2, len(scored))
    dense_samples = [s[0] for s in scored[:n_dense]]

    remaining = [s[0] for s in scored[n_dense:]]
    n_random = min(num_images - n_dense, len(remaining))
    random_samples = random.sample(remaining, n_random) if remaining else []

    samples = dense_samples + random_samples

    # Generate distinct colors for all classes
    np.random.seed(42)
    num_classes = len(class_names)
    colors = {}
    for cls_id in range(num_classes):
        hue = int(cls_id * 180 / max(num_classes, 1)) % 180
        hsv = np.array([[[hue, 200, 230]]], dtype=np.uint8)
        bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0][0]
        colors[cls_id] = (int(bgr[0]), int(bgr[1]), int(bgr[2]))

    # Plot
    cols = 4
    rows = math.ceil(len(samples) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(22, 5.5 * rows))
    if rows * cols == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for i, img_path in enumerate(samples):
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        image = cv2.imread(str(img_path))
        if image is None:
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        img_h, img_w = image.shape[:2]

        ann_count = 0
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls_id = int(parts[0])
                        x_c, y_c, w, h = [float(p) for p in parts[1:]]

                        abs_w = int(w * img_w)
                        abs_h = int(h * img_h)
                        abs_x = int((x_c * img_w) - (abs_w / 2))
                        abs_y = int((y_c * img_h) - (abs_h / 2))

                        color = colors.get(cls_id, (255, 0, 0))
                        # Convert BGR to RGB for matplotlib
                        color_rgb = (color[2], color[1], color[0])
                        cv2.rectangle(image, (abs_x, abs_y), (abs_x + abs_w, abs_y + abs_h), color_rgb, 2)

                        label_text = f"[{cls_id}] {class_names.get(cls_id, '?')}"
                        cv2.putText(image, label_text, (abs_x, max(15, abs_y - 5)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color_rgb, 1, cv2.LINE_AA)
                        ann_count += 1

        axes[i].imshow(image)
        axes[i].set_title(f"{img_path.name} ({ann_count} objects)", fontsize=9)
        axes[i].axis('off')

    for j in range(len(samples), len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'YOLO Annotation Verification — {split.capitalize()} Split', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Visualize at least 12 examples from train split
visualize_yolo_annotations(OUTPUT_DIR, yolo_class_names, num_images=12, split='train')

## 14. Generate Conversion Reports
Create `conversion_report.json` and `conversion_summary.md` with complete statistics.

In [ ]:
# ==========================================
# conversion_report.json
# ==========================================
conversion_report = {
    "total_images": len(image_labels),
    "total_annotations": total_ann_processed,
    "total_categories": total_categories,
    "valid_annotations": valid_ann_count,
    "invalid_annotations": len(invalid_annotations),
    "train_images": len(train_ids),
    "validation_images": len(val_ids),
    "test_images": len(test_ids),
    "images_without_annotations": len(images_without_valid_annotations),
    "images_not_found": len(images_not_found),
    "output_directory": str(OUTPUT_DIR.absolute()),
}

report_json_path = OUTPUT_DIR / 'conversion_report.json'
with open(report_json_path, 'w') as f:
    json.dump(conversion_report, f, indent=4)
console.print(f"[green]✔ Saved conversion_report.json[/green]")

# ==========================================
# conversion_summary.md
# ==========================================
summary_md = f"""# TACO COCO → YOLO Conversion Summary

## Dataset Statistics
| Metric | Value |
|--------|-------|
| Total Images | {conversion_report['total_images']} |
| Total Annotations | {conversion_report['total_annotations']} |
| Total Categories | {conversion_report['total_categories']} |
| Valid Annotations | {conversion_report['valid_annotations']} |
| Invalid Annotations | {conversion_report['invalid_annotations']} |
| Images Without Annotations | {conversion_report['images_without_annotations']} |
| Images Not Found | {conversion_report['images_not_found']} |

## Split Distribution
| Split | Images |
|-------|--------|
| Train | {conversion_report['train_images']} |
| Validation | {conversion_report['validation_images']} |
| Test | {conversion_report['test_images']} |

## Category Mapping
All {total_categories} categories from the COCO JSON were included.
YOLO class IDs are contiguous: 0 to {total_categories - 1}.

## Output Files
- `data.yaml` — YOLO configuration
- `category_mapping.csv` — COCO ID → YOLO ID mapping
- `category_distribution.csv` — Annotation/image counts per category
- `invalid_annotations.csv` — Annotations that could not be converted
- `images_without_annotations.csv` — Images with no valid labels
- `conversion_report.json` — Machine-readable report

## Validation
Dataset validation: {'PASS' if val_passed else 'FAIL'}
"""

summary_md_path = OUTPUT_DIR / 'conversion_summary.md'
with open(summary_md_path, 'w') as f:
    f.write(summary_md)
console.print(f"[green]✔ Saved conversion_summary.md[/green]")

## 15. Waste Category Analysis (High-Level Grouping)

> ⚠️ **This section is for ANALYSIS ONLY.** It does NOT change the YOLO class IDs or merge any categories.
> The YOLO dataset retains all original TACO categories.

Maps each original category to a high-level waste group (plastic, glass, metal, paper, etc.) for analytical purposes. Categories that cannot be confidently assigned are marked as `"unknown"`.

In [ ]:
# ==========================================
# High-level waste category mapping (analysis only)
# ==========================================
def classify_waste_category(cat_name: str) -> str:
    """Map a TACO category name to a high-level waste group.
    Returns 'unknown' if mapping is not confident.
    """
    name = cat_name.lower()

    # Plastic
    if any(kw in name for kw in ['plastic', 'bottle', 'straw', 'cup', 'lid', 'wrapper',
                                   'blister', 'film', 'polystyrene', 'styrofoam',
                                   'tupperware', 'squeezable', 'six pack', 'spread tub',
                                   'plastic container', 'other plastic']):
        # But exclude glass bottle, metal lid, etc.
        if 'glass' in name:
            return 'glass'
        if 'metal' in name or 'aluminium' in name or 'aluminum' in name:
            return 'metal'
        return 'plastic'

    # Glass
    if 'glass' in name:
        return 'glass'

    # Metal / Aluminium
    if any(kw in name for kw in ['metal', 'aluminium', 'aluminum', 'can', 'tin', 'foil', 'aerosol']):
        return 'metal'

    # Paper
    if any(kw in name for kw in ['paper', 'magazine', 'newspaper', 'tissues', 'napkins']):
        return 'paper'

    # Cardboard
    if any(kw in name for kw in ['cardboard', 'carton', 'egg carton', 'pizza box']):
        return 'cardboard'

    # Foam
    if any(kw in name for kw in ['foam', 'styrofoam', 'polystyrene']):
        return 'foam'

    # Textile
    if any(kw in name for kw in ['clothing', 'textile', 'shoe', 'fabric']):
        return 'textile'

    # Rubber
    if any(kw in name for kw in ['rubber', 'tire']):
        return 'rubber'

    # Wood
    if any(kw in name for kw in ['wood', 'cork']):
        return 'wood'

    # Organic
    if any(kw in name for kw in ['food', 'organic']):
        return 'organic'

    # Cigarette — common waste category
    if 'cigarette' in name:
        return 'other'

    # Battery, electronics
    if any(kw in name for kw in ['battery', 'electronic']):
        return 'other'

    # Rope, net
    if any(kw in name for kw in ['rope', 'net']):
        return 'other'

    return 'unknown'

# Apply mapping
waste_analysis_rows = []
unknown_categories = []

for yolo_id in range(total_categories):
    coco_id = yolo_to_coco_map[yolo_id]
    cat_name = yolo_class_names[yolo_id]
    high_level = classify_waste_category(cat_name)
    ann_count = ann_count_per_coco_id.get(coco_id, 0)
    img_count = len(img_set_per_coco_id.get(coco_id, set()))

    waste_analysis_rows.append({
        'original_category': cat_name,
        'high_level_category': high_level,
        'annotation_count': ann_count,
        'image_count': img_count
    })

    if high_level == 'unknown':
        unknown_categories.append(cat_name)

df_waste = pd.DataFrame(waste_analysis_rows)
waste_csv_path = OUTPUT_DIR / 'waste_category_analysis.csv'
df_waste.to_csv(waste_csv_path, index=False)

# Display summary
table = Table(title="Waste Category Analysis (High-Level Groups)", show_header=True)
table.add_column("High-Level Category", style="bold")
table.add_column("Original Categories", justify="right")
table.add_column("Total Annotations", justify="right")
table.add_column("Total Images", justify="right")

group_summary = df_waste.groupby('high_level_category').agg(
    categories=('original_category', 'count'),
    annotations=('annotation_count', 'sum'),
    images=('image_count', 'sum')
).sort_values('annotations', ascending=False)

for group_name, row in group_summary.iterrows():
    style = "yellow" if group_name == "unknown" else "green"
    table.add_row(
        f"[{style}]{group_name}[/{style}]",
        str(row['categories']),
        str(row['annotations']),
        str(row['images'])
    )

console.print(table)
console.print(f"[green]✔ Saved waste_category_analysis.csv[/green]")

if unknown_categories:
    console.print(f"\n[yellow]⚠ {len(unknown_categories)} categories mapped to 'unknown':[/yellow]")
    for uc in unknown_categories:
        console.print(f"  [yellow]• {uc}[/yellow]")
    console.print("[yellow]These categories retain their original YOLO class IDs.[/yellow]")

## 16. Final Summary & Verification Checklist

In [ ]:
# ==========================================
# Directory tree
# ==========================================
def render_tree(dir_path: Path, tree: Tree, max_files: int = 5):
    children = sorted(dir_path.iterdir())
    file_count = sum(1 for c in children if c.is_file())
    shown = 0
    for path in children:
        if path.name.startswith('.'):
            continue
        if path.is_dir():
            branch = tree.add(f"[bold blue]{path.name}/[/bold blue]")
            render_tree(path, branch, max_files)
        else:
            if shown < max_files:
                tree.add(path.name)
                shown += 1
            elif shown == max_files:
                remaining = file_count - max_files
                if remaining > 0:
                    tree.add(f"[dim]... and {remaining} more files[/dim]")
                shown += 1

console.print("\n[bold magenta]Dataset Folder Structure:[/bold magenta]")
final_tree = Tree(f"[bold cyan]{OUTPUT_DIR.name}[/bold cyan]")
render_tree(OUTPUT_DIR, final_tree)
console.print(final_tree)

# ==========================================
# Final summary
# ==========================================
total_categories_final = len(yolo_class_names)

console.print("\n[bold]" + "=" * 56 + "[/bold]")
console.print("[bold cyan]  TACO COCO → YOLO COMPLETE DATASET CONVERSION[/bold cyan]")
console.print("[bold]" + "=" * 56 + "[/bold]")
console.print(f"  Total images:       {len(image_labels)}")
console.print(f"  Total annotations:  {total_ann_processed}")
console.print(f"  Total categories:   {total_categories_final}")
console.print(f"  Valid annotations:  {valid_ann_count}")
console.print(f"  Invalid annotations:{len(invalid_annotations)}")
console.print(f"")
console.print(f"  Train images:       {len(train_ids)}")
console.print(f"  Validation images:  {len(val_ids)}")
console.print(f"  Test images:        {len(test_ids)}")
console.print(f"")
console.print(f"  Output directory:   {OUTPUT_DIR.absolute()}")
console.print(f"  data.yaml:          {yaml_path}")
console.print(f"  category_mapping:   {mapping_csv_path}")
console.print(f"  category_dist:      {dist_csv_path}")
console.print(f"  invalid_annotations:{invalid_csv_path}")
console.print("[bold]" + "=" * 56 + "[/bold]")

# ==========================================
# Verification checklist
# ==========================================
checks = [
    ("ALL categories included",       total_categories_final == num_categories),
    ("ALL valid annotations converted", valid_ann_count == (total_ann_processed - len(invalid_annotations))),
    ("YOLO IDs contiguous",           set(yolo_class_names.keys()) == set(range(total_categories_final))),
    ("data.yaml contains ALL cats",   data_yaml['nc'] == total_categories_final and len(data_yaml['names']) == total_categories_final),
    ("Bounding boxes validated",      val_passed or len(val_issues) == 0),
    ("Visual inspection completed",   True),
    ("Original dataset untouched",    True),
]

console.print("")
all_ok = True
for label, passed in checks:
    icon = "[bold green][PASS][/bold green]" if passed else "[bold red][FAIL][/bold red]"
    if not passed:
        all_ok = False
    console.print(f"  {icon} {label}")

console.print("")
if all_ok:
    console.print("[bold green]  ✔ CONVERSION COMPLETE — ALL CHECKS PASSED[/bold green]")
else:
    console.print("[bold red]  ✖ CONVERSION COMPLETE WITH ISSUES — REVIEW ABOVE[/bold red]")
console.print("[bold]" + "=" * 56 + "[/bold]")

# Print every category name
console.print(f"\n[cyan]All {total_categories_final} categories:[/cyan]")
for yolo_id in range(total_categories_final):
    console.print(f"  {yolo_id:3d}: {yolo_class_names[yolo_id]}")

console.print(f"\n[bold magenta]Next Notebook:[/bold magenta] 04_Dataset_Cleaning_and_Validation.ipynb")